In [ ]:
import os

import numpy as np
from matplotlib import pyplot as plt
from ase.visualize import view
import nqetools as nqe
# This follows:
# https://github.com/i-pi/piqm2023-tutorial/blob/main/05-RPI/tutorial-4.ipynb

In [ ]:
# Paths
directory_opti = 'opti'
directory_phonon_react = 'phonon_react'
directory_ts = 'ts'
directory_phonon_ts = 'phonon_ts'
directory_instanton = 'instanton'

# Driver
driver_code = 'ase-mace'

# Values
temperature = 300.0
n_beads = 10
tol_energy = 5.0e-4
tol_force = 5.0e-4
tol_position = 1.0e-4
total_steps = 1000
optimizer = "cg"


In [ ]:
atoms = nqe.read_ipi_xyz("react.xyz")[-1]
atoms.center(vacuum=10.0)
view(atoms)

In [ ]:
from ase.data.pubchem import pubchem_atoms_search
atoms = pubchem_atoms_search(smiles='O=CO')

atoms.center(vacuum=0.0)
atoms = nqe.align_principal_axis(atoms, axis='z')
atoms = nqe.make_dimer(atoms, translate=[2.4, 6.0, 0.0])
atoms.center(vacuum=10.0)
view(atoms)

prod_1 = nqe.swap_bonding_configuration(atoms, 0, 4, 6)
prod = nqe.swap_bonding_configuration(prod_1, 5, 9, 1)
view(prod)

In [ ]:
import geodesic_interpolate as gi

n_images = 11

atoms_ts = gi.geodesic_interpolate([atoms, prod], n_images=n_images)

# Select the center image as the transition state
atoms_ts = atoms_ts[n_images//2]
view(atoms_ts)


In [ ]:
# Run minimisation
output = nqe.run_optimise(directory_opti,
                          atoms,
                          driver=driver_code,
                          tol_energy=tol_energy,
                          tol_force=tol_force,
                          tol_position=tol_position)
atoms_opti, output_data_opti, output_desc_opti = output

In [ ]:
view(atoms_opti)

In [ ]:
# Plot the energy of the minimisation
nqe.plot_step_energy(output_data_opti, save=False)

In [ ]:
nqe.run_phonons(directory_phonon_react,
                atoms_opti,
                driver=driver_code)

In [ ]:
eigvals = np.genfromtxt(os.path.join(directory_phonon_react, 'phonon.phonons.eigval'))
fr = nqe.freq_from_eigvals(eigvals)

plt.plot(fr, 'o')
plt.xlabel('Vibrational Mode  Index')
plt.ylabel('Frequency (cm$^{-1}$)')
plt.show()

In [ ]:
# n_atoms = len(atoms)
# atoms_ts = nqe.read_ipi_xyz("ts.xyz")[-1]

In [ ]:
output = nqe.run_ts(directory_ts,
                    atoms_ts,
                    driver=driver_code,
                    tol_energy=tol_energy,
                    tol_force=tol_force,
                    tol_position=tol_position)
atoms_ts, output_data_ts, output_desc_ts = output

In [ ]:
# Plot the energy of the minimisation
nqe.plot_step_energy(output_data_ts, save=False)

In [ ]:
rate = nqe.calc_forward_rate(directory_phonon_react,
                             directory_ts,
                             temperature,
                             filter_list=n_atoms - 1)
print('Reaction rate = {}'.format(rate), flush=True)

In [ ]:
nqe.run_phonons(directory_phonon_ts,
                atoms_ts,
                driver=driver_code)

In [ ]:
eigvals = np.genfromtxt(os.path.join(directory_phonon_ts, 'phonon.phonons.eigval'))
fr = nqe.freq_from_eigvals(eigvals)

plt.plot(fr, 'o')
plt.xlabel('Vibrational Mode  Index')
plt.ylabel('Frequency (cm$^{-1}$)')
plt.show()

In [ ]:
# Run the instanton
nqe.run_instanton(directory_instanton,
                  atoms_ts,
                  directory_ts,
                  driver=driver_code,
                  n_beads=n_beads,
                  temperature=temperature,
                  tol_energy=tol_energy,
                  tol_force=tol_force,
                  tol_position=tol_position)

In [ ]:
kappa = nqe.calc_kappa_full(directory_phonon_react,
                            directory_ts,
                            directory_instanton,
                            temperature,
                            n_beads,
                            filter_list=n_atoms - 1)
print('Tunneling factor, kappa = {:5.5f}'.format(kappa), flush=True)

In [ ]:
# Converge over temperature
list_temperature = [200, 250, 300, 350, 400]
list_kappa = []
n_beads = 40

for temperature in list_temperature:
    print(f"Running with {temperature} K")
    nqe.run_instanton(directory_instanton,
                      atoms_ts,
                      directory_ts,
                      driver=driver_code,
                      n_beads=n_beads,
                      temperature=temperature,
                      tol_energy=tol_energy,
                      tol_force=tol_force,
                      tol_position=tol_position, )

    kappa = nqe.calc_kappa_full(directory_phonon_react,
                                directory_ts,
                                directory_instanton,
                                temperature,
                                n_beads,
                                filter_list=n_atoms - 1)
    list_kappa.append(kappa)

In [ ]:
print(list_kappa)
plt.plot(list_temperature, list_kappa, 'o-')
plt.xlabel('Temperature (K)')
plt.ylabel('Kappa')
plt.show()

In [ ]:
nqe.plot_kappa_temperature(list_temperature, list_kappa, save=False)
nqe.plot_kappa_temperature_inv(list_temperature, list_kappa, save=False)

In [ ]:
rates = []
for temperature in list_temperature:
    output = nqe.run_ts(directory_ts,
                        atoms_ts,
                        driver=driver_code,
                        tol_energy=tol_energy,
                        tol_force=tol_force,
                        tol_position=tol_position,
                        total_steps=total_steps)

    atoms_ts, output_data_ts, output_desc_ts = output

    rate = nqe.calc_forward_rate(directory_phonon_react,
                                 directory_ts,
                                 temperature,
                                 filter_list=n_atoms - 1)

    rates.append(rate)

In [ ]:
nqe.plot_arrhenius(list_temperature, rates)

rates_quantum = []
for i, rate in enumerate(rates):
    rates_quantum.append(rate * list_kappa[i])

nqe.plot_arrhenius_2(list_temperature, rates, rates_quantum, save=False)

In [ ]:
# Try and converge over beads
list_n_beads = [2, 4, 8, 10, 20, 40, 60, 80]
list_kappa = []

for n_beads in list_n_beads:
    print(f"Running with {n_beads} beads")
    nqe.run_instanton(directory_instanton,
                      atoms_ts,
                      directory_ts,
                      driver=driver_code,
                      n_beads=n_beads,
                      temperature=temperature,
                      tol_energy=tol_energy,
                      tol_force=tol_force,
                      tol_position=tol_position)
    kappa = nqe.calc_kappa_full(directory_phonon_react,
                                directory_ts,
                                directory_instanton,
                                temperature,
                                n_beads,
                                filter_list=n_atoms - 1)
    list_kappa.append(kappa)

In [ ]:
nqe.plot_bead_convergence(list_n_beads, list_kappa)

In [ ]:
nqe.remove_directory(directory_opti)
nqe.remove_directory(directory_phonon_react)
nqe.remove_directory(directory_ts)
nqe.remove_directory(directory_phonon_ts)
nqe.remove_directory(directory_instanton)